In [59]:
import pandas as pd
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from tqdm.auto import tqdm

In [60]:
# Load data
df = pd.read_csv("files/back_translation_a1_a2.csv")

In [61]:
df

,original_english,translated_welsh,back_translated_english,cefr_level,BLEU,chrF,cosine_similarity
0,The late 19th century marks the start of psych...,Ym 19egrodd y bedwaredd ganrif ar bymtheg o bw...,In 19th century the late 1800 ' s of the late ...,A2,6.92,26.57,0.4495
1,Cartridge paper is the basic type of drawing p...,Constellation name (optional),FIVE FEUDAL LORDS,A2,0.00,0.00,-0.0948
2,This is called the observer effect.,Mae hyn yn enw' r effaith.,This is the name of the effect.,A2,16.52,35.91,0.7631
3,This was originally intended to provide a chec...,Roedd hyn yn bwriadu darparu grym gwleidyddol ...,This meant to provide political power on polit...,A2,26.58,47.73,0.7600
4,Different policies were applied in Albania and...,Cafodd pob paid a oedd yn fwy o'r Undeb Sofiet...,All of the Soviet Union made more the Soviet U...,A2,15.22,24.89,0.4676
...,...,...,...,...,...,...,...
2521,"The Elephant Show\r\n\r\nby Daniel Allsop, age...","Mae'r hawyr yn gwisgo gan Daniel bob oedran, 1...","The haus was dressed by Daniel's age, 14, I we...",A2,0.07,7.90,0.3191
2522,Mount Kilimanjaro\r\n\r\nMount Kilimanjaro is ...,(2 Bren. 15: 6 - 8) Er bod Mynyddmanman Americ...,Although Mountmanman America were beyond the h...,A2,0.03,9.62,0.4589
2523,Visit the Edinburgh Festival!\r\n\r\nEvery yea...,"Ewch i'r ddesbys, Bob flwyddyn, mae miloedd o ...","Go to the last year, Each year during this yea...",A2,0.10,7.64,0.2307
2524,The Rhino\r\n\r\nThere are five different type...,Mae'r cannoedd o bump yn cynnwys mathau gwahan...,The hundreds of five types of different types ...,A2,0.05,11.47,0.2665


In [62]:
# Load CEFR model
model_path = "UniversalCEFR/EuroBERT-210m-cefr-all-classifier"
model = AutoModelForSequenceClassification.from_pretrained(model_path, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True, trust_remote_code=True)

In [63]:
#Classification pipeline
cefr_pipeline = pipeline("text-classification", model=model, tokenizer=tokenizer,trust_remote_code=True)

Device set to use cpu


In [64]:
# Map model output label to CEFR level
id2label = {i: lvl for i, lvl in enumerate(["A1", "A2", "B1", "B2", "C1", "C2"])}
def label_to_cefr(label):
    idx = int(label.split("_")[1])
    return id2label[idx]

In [65]:
# Prediction function
def predict_cefr(texts):
    raw_outputs = cefr_pipeline(texts, truncation=True, padding=True)
    return [pred['label'] for pred in raw_outputs]

In [66]:
# Apply to your columns
text_cols = ['translated_welsh', 'back_translated_english']
for col in text_cols:
    df[col + '_predicted_cefr'] = predict_cefr(df[col].astype(str).tolist())

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [67]:
#Save to CSV
df.to_csv("files/fineTuned_cefr_predictions.csv", index=False)

In [68]:
#High-similarity pairs
filtered_df = df[
    (df["BLEU"] > 20) &
    (df["chrF"] > 40) &
    (df["cosine_similarity"] > 0.8)
]

#Save CSV
filtered_df.to_csv("files/fineTuned_filtered_high_similarity.csv", index=False)

In [69]:
filtered_df

,original_english,translated_welsh,back_translated_english,cefr_level,BLEU,chrF,cosine_similarity,translated_welsh_predicted_cefr,back_translated_english_predicted_cefr
6,"And the little prince went away, puzzled.","Ac mae'r tywysog yn mynd i ffwrdd, yn ddryslyd.","And the prince goes away, puzzled.",A1,37.71,58.39,0.8893,A1,A1
7,"This is, to me, the loveliest and saddest land...","Dyma, i mi, y mae'r cariad yn drist a'r lleafa...","This is, for me, that love is sad and the plac...",A2,22.79,42.49,0.8236,A2,B1
8,The fifth planet was very strange.,Roedd y pumed yn rhyfedd iawn.,The fifth was very strange.,A2,51.15,70.56,0.8031,A2,A2
14,"To be honest, by now I didn't really care.","I fod yn onest, erbyn hyn doeddwn i ddim wir w...","To be honest, I did not really care.",A2,35.54,67.51,0.8883,A2,A2
15,This film does have its good points.,Mae'r ffilm yn cyfeirio at ei pwyntiau da.,The movie refers to his good points.,A2,20.56,43.83,0.8424,A2,B1
...,...,...,...,...,...,...,...,...,...
2306,3 ... Bf5 is most often played .,3 ... B5 yn aml yn chwarae.,3 ... B5 often play.,A2,29.80,43.85,0.8461,A2,A2
2307,Exchange students from Italy and Germany help ...,Mae myfyrwyr o'r Eidal a'r Almaen yn helpu Sae...,Students from Italy and Germany helped English...,A2,30.98,66.85,0.9199,A2,A2
2309,"After one year , she opened the bookshop again .","Ar ôl un flwyddyn, agorodd y llyfrau hedfan eto.","After one year, he opened the books again.",A2,35.49,76.52,0.8847,A2,A2
2325,"She was an only child , born in London .","Roedd hi'n unig blentyn, geni yn Llundain.","She was only a child, born in London.",A2,59.88,76.34,0.9979,A2,A2
